In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from datetime import datetime, timedelta

ORIGINAL_LOG = "runs/ricotta-2.0/r1/ricotta_2_0/train.log"
WSD20_LOG = "runs/ricotta-2.0/r1-wsd20(minlr0.1)/ricotta_2_0/train.log"
BRANCH_STEP = 12_500


def load_log(path):
    train_rows, val_rows = [], []

    with open(path, "r") as f:
        for line in f:
            parts = line.split()
            if len(parts) < 2 or not parts[0].isdigit():
                continue

            step = int(parts[0])
            kind = parts[1]

            values = {
                k: float(v)
                for k, v in re.findall(r"(\w+)=([\d.eE+-]+)", line)
            }

            if kind == "train":
                train_rows.append({"step": step, **values})
            elif kind == "val":
                val_rows.append({"step": step, **values})

    return pd.DataFrame(train_rows), pd.DataFrame(val_rows)


train, val = load_log(ORIGINAL_LOG)
train20, val20 = load_log(WSD20_LOG)

train20 = train20[train20["step"] >= BRANCH_STEP].copy()
val20 = val20[val20["step"] >= BRANCH_STEP].copy()


def print_stats(name, train, val):
    print(f"\n{name}")

    if len(train):
        min_idx = train["loss"].idxmin()
        print(f"Steps:             {len(train):,}")
        print(f"Epoch:             {train['epoch'].iloc[-1]:.4f}")
        print(f"Tokens:            {train['tokens'].iloc[-1] / 1e9:.3f}B")
        print(f"Lowest train loss: {train.loc[min_idx, 'loss']:.6f} at step {int(train.loc[min_idx, 'step']):,}")

    if len(val):
        min_idx = val["loss"].idxmin()
        print(f"Lowest val loss:   {val.loc[min_idx, 'loss']:.6f} at step {int(val.loc[min_idx, 'step']):,}")


print_stats("Original — 10% WSD", train, val)
print_stats("WSD20 — 20% WSD", train20, val20)

FileNotFoundError: [Errno 2] No such file or directory: 'runs/ricotta-2.0/r1/ricotta_2_0/train.log'

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Raw training loss
ax.plot(
    train["step"],
    train["loss"],
    linewidth=0.5,
    alpha=0.18,
    label="10% WSD — train",
)

ax.plot(
    train20["step"],
    train20["loss"],
    linewidth=0.5,
    alpha=0.25,
    label="20% WSD — train",
)

# Smoothed training loss
window = 100

smooth = train["loss"].rolling(window, min_periods=1).mean()
smooth20 = train20["loss"].rolling(window, min_periods=1).mean()

ax.plot(
    train["step"],
    smooth,
    linewidth=2,
    label="10% WSD — 100-step average",
)

ax.plot(
    train20["step"],
    smooth20,
    linewidth=2,
    label="20% WSD — 100-step average",
)

# Validation
if len(val):
    ax.plot(
        val["step"],
        val["loss"],
        marker="o",
        markersize=5,
        linewidth=2,
        label="10% WSD — validation",
    )

if len(val20):
    ax.plot(
        val20["step"],
        val20["loss"],
        marker="o",
        markersize=5,
        linewidth=2,
        label="20% WSD — validation",
    )

ax.axvline(
    BRANCH_STEP,
    linestyle="--",
    linewidth=1.2,
    alpha=0.6,
    label="Branch at step 12,500",
)

ax.set_xlabel("Training Step")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Ricotta-2.0 — 10% vs 20% WSD Decay")
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=3, top=4)
ax.legend()

plt.show()